# 3. Memory

## 학습 목표

1. LLM이 이전 대화를 스스로 기억하지 못하는 이유를 설명할 수 있습니다.
2. 대화 기록을 직접 누적하는 Buffer 방식을 구현합니다.
3. LangChain 1.x 방식(LangGraph 체크포인터)으로 **사용자별 대화 기록**을 관리합니다.
4. Window(최근 N개 유지)와 Summary(요약) 전략으로 토큰 사용량을 줄입니다.
5. 토큰 사용량과 예상 비용을 확인합니다.

**기준 버전**: LangChain 1.x, LangGraph 1.x, 모델 `gpt-6-luna`

> **버전 변경 사항**
> - `ConversationBufferMemory` 같은 예전 Memory 클래스는 LangChain 1.x 표준에서 빠졌습니다.
> - `RunnableWithMessageHistory`도 `langchain-core` 1.3.3부터 **deprecated**(2.0에서 제거 예정)입니다.
> - 현재 권장 방식은 **LangGraph의 체크포인터(checkpointer)** 로 대화 기록을 저장하는 것입니다. 이 노트북은 이 방식으로 바꿔 설명합니다.

## 1. 왜 LLM은 과거 대화를 "자동으로" 기억하지 못할까?

- LLM은 **상태가 없는(stateless)** 모델입니다. 대화 기록을 저장하는 공간이 없습니다.
- 호출될 때마다 **현재 요청에 포함된 입력(메시지)만** 보고 답합니다.
- 따라서 이전 대화를 이어 가려면 **과거 대화 기록을 다시 입력에 포함**해야 합니다.

## 2. LangChain Memory란?

- 이전 대화 내용을 **저장(store)** 하고, 다음 요청 때 입력에 **주입(inject)** 하는 구조입니다.
- 사용자는 AI가 "기억하는 것처럼" 느끼지만, 실제로는 매 호출마다 과거 대화를 프롬프트에 붙여 다시 보내는 방식입니다.
- LLM 내부 기능이 아니라 LangChain 실행 계층에서 관리하는 상태 관리 시스템입니다.

| 전략 | 특징 | 사용 목적 | 비고 | LangChain 1.x 구현 |
|---|---|---|---|---|
| Buffer | 모든 대화를 그대로 저장 | 짧은 대화, 디버깅 | 정확하지만 비용 증가 | 메시지 리스트 / 체크포인터 |
| Window | 최근 N개만 유지 | 실시간 채팅, 효율성 | 저렴하지만 과거 망각 | `trim_messages` |
| Summary | 이전 대화를 요약해 유지 | 장기 대화 | 효율적이지만 요약 과정에서 정보 손실·추가 비용 | `SummarizationMiddleware` |

### 대화 기록(Message History)의 기본 형태

```text
[SystemMessage]  "당신은 친절한 AI 강사입니다."
      ↓
[HumanMessage]   "오늘 수업 주제가 무엇인가요?"        (Human #1)
      ↓
[AIMessage]      "오늘은 랭체인 메모리에 대해 배웁니다." (AI #1)
      ↓
[HumanMessage]   "쉽게 설명해 주세요."                  (Human #2)
      ↓
[AIMessage]      "비유로 차근차근 알려 드릴게요."        (AI #2)
      ↓
[HumanMessage]   "그럼 첫 번째 개념부터 알려 주세요!"    (새 질문)
```

새 질문을 보낼 때 LLM이 받는 입력은 **[과거 기록 전체 + 새 질문]** 입니다.

## 3. 전략(Memory)과 저장소(History)의 구분

| 구분 | 역할 | LangChain 1.x에서 사용하는 것 |
|---|---|---|
| 전략 (Memory) | 기록을 **어떻게** 프롬프트에 넣을지 결정 (전부 / 최근 N개 / 요약) | `trim_messages`, `SummarizationMiddleware` |
| 저장소 (History) | 메시지가 **실제로 쌓이는 곳** | 파이썬 리스트, `InMemoryChatMessageHistory`, 체크포인터(`InMemorySaver` 등) |

Buffer, Window, Summary는 **전략**의 종류이고, 데이터를 어디에 저장할지는 **저장소**의 영역입니다.

### 실습 환경 준비

터미널에서 필요한 패키지를 설치하고, 프로젝트 최상위 폴더의 `.env` 파일에 API 키와 모델명을 저장합니다.

```text
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
OPENAI_DEFAULT_MODEL="gpt-6-luna"
```

> API 키는 코드 셀이나 GitHub에 직접 입력하지 마세요.

```bash
uv add langchain langchain-openai langgraph python-dotenv
```

In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")

# reasoning_effort="none": 수업 예제는 빠른 응답과 낮은 비용을 위해 추론 기능을 끕니다.
llm = ChatOpenAI(model=default_model, reasoning_effort="none")
print("사용 모델:", default_model)

> **gpt-6-luna 사용 시 참고**
> - `reasoning_effort`는 `none` / `low` / `medium`(기본) / `high` / `xhigh` / `max`를 지원합니다. 강의 예제는 `none`으로 통일합니다.
> - `temperature`는 추론이 꺼져 있을 때(`reasoning_effort="none"`)만 적용됩니다.
> - 모델명·요금·지원 파라미터는 바뀔 수 있으므로 수업 전에 [OpenAI 모델 문서](https://developers.openai.com/api/docs/models/gpt-6-luna)를 확인하세요.

## 4. Buffer Memory - 대화를 그대로 누적하기

맥락이 이어지도록 LLM을 3번 호출합니다. 매번 **지금까지의 메시지 전체**를 다시 보냅니다.

```text
1번 호출: system + user1
2번 호출: system + user1 + ai1 + user2
3번 호출: system + user1 + ai1 + user2 + ai2 + user3
```

In [ ]:
messages = [
    ("system", "당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다."),
    ("human", "부산 여행에서 딱 한 곳만 가 봐야 한다면 어떤 곳인지 알려 주세요."),
]

response = llm.invoke(messages)
messages.append(("ai", response.content))   # 답변을 기록에 추가

print(response.content)

In [ ]:
messages.append(("human", "부산역에서 그곳에 가는 교통편을 알려 주세요."))

response = llm.invoke(messages)
messages.append(("ai", response.content))

print(response.content)

In [ ]:
messages.append(("human", "그 근처에서 먹을 만한 음식점들을 추천해 주세요."))

response = llm.invoke(messages)
messages.append(("ai", response.content))

print(response.content)
print("\n누적 메시지 수:", len(messages))

### 실습하기

위 3번의 호출을 **반복문** 하나로 묶어 실행해 보세요.

```python
questions = [
    "부산 여행에서 딱 한 곳만 가 봐야 한다면 어떤 곳인지 알려 주세요.",
    "부산역에서 그곳에 가는 교통편을 알려 주세요.",
    "그 근처에서 먹을 만한 음식점들을 추천해 주세요.",
]
messages = [("system", "당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다.")]
```

In [ ]:
# 여기에 실습 코드를 작성하세요.

### LLM의 한계 - 토큰 제한(Context Window)과 비용

- 모든 LLM은 한 번에 처리할 수 있는 입력·출력의 최대 크기(**Context Window**)가 정해져 있습니다.
  (`gpt-6-luna`는 약 105만 토큰으로 매우 크지만, 한계가 있다는 점은 같습니다.)
- 대화 기록이 쌓일수록 **매 호출마다 보내는 토큰이 늘어나고, 비용도 비례해서 증가**합니다.
- Buffer 방식은 가장 단순하고 정확하지만, 대화가 길어지면 비용과 응답 속도에 부담이 됩니다.
- 그래서 **최근 N개만 남기는 Window** 나 **내용을 압축하는 Summary** 전략이 필요합니다.

## 5. InMemoryChatMessageHistory - 메시지 저장소

- 메시지를 역할 객체(`SystemMessage`, `HumanMessage`, `AIMessage`) 그대로 **리스트로 저장**하는 기본 저장소입니다.
- 서버 RAM에 저장되므로 프로그램이 종료되면 기록이 사라집니다(휘발성).
- 실제 서비스에서는 Redis, PostgreSQL 같은 외부 저장소로 바꿔 사용합니다.

| 메서드 | 역할 |
|---|---|
| `add_message()` | 메시지 추가 |
| `add_user_message()` | 사용자 메시지 추가 |
| `add_ai_message()` | AI 메시지 추가 |
| `.messages` | 지금까지의 대화 기록(리스트) |

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import SystemMessage

memory = InMemoryChatMessageHistory()

memory.add_message(SystemMessage(content="당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다."))
memory.add_user_message("부산 여행에서 딱 한 곳만 가 봐야 한다면 어떤 곳인지 알려 주세요.")

response = llm.invoke(memory.messages)
memory.add_ai_message(response.content)

memory.add_user_message("부산역에서 그곳에 가는 교통편을 알려 주세요.")
response = llm.invoke(memory.messages)
memory.add_ai_message(response.content)

for msg in memory.messages:
    print(f"[{msg.type}] {msg.content[:60]}")

## 6. 사용자별 대화 기록 관리 - 체크포인터(checkpointer)

여러 사용자가 동시에 대화해도 기록이 섞이지 않으려면 **사용자(대화방)별로 기록을 분리**해야 합니다.
LangChain 1.x에서는 `create_agent`에 **체크포인터**를 연결하고, 호출할 때 `thread_id`(대화방 번호)를 넘기면 됩니다.

| 예전 방식 (`RunnableWithMessageHistory`) | 현재 방식 (LangGraph 체크포인터) |
|---|---|
| `store = {}` + `get_history()` 함수 직접 작성 | `InMemorySaver()` 한 줄 |
| `MessagesPlaceholder("history")` 직접 배치 | 자동으로 기록을 붙여서 전달 |
| `session_id` | `thread_id` |
| `input_messages_key`, `history_messages_key` 이름 맞추기 | 필요 없음 |

```text
1. 사용자 질문 + thread_id="1"
2. 체크포인터에서 thread "1"의 기록 조회 (없으면 새로 생성)
3. [system + 과거 기록 + 새 질문]을 LLM에 전달
4. 이번 턴의 질문과 답변을 thread "1"에 자동 저장
5. 같은 thread_id로 다음 질문 → 2번부터 반복
```

> 비유: 체크포인터는 **아파트 단지의 우편함 관리실**, `thread_id`는 **동·호수**입니다. "101호 우편함 주세요" 하면 없을 때는 새로 만들고, 있으면 기존 우편함을 꺼내 줍니다.

> `create_agent`는 **7. Agent** 에서 자세히 배웁니다. 여기서는 도구 없이(`tools=[]`) 대화 기록 관리 용도로만 사용합니다.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

chatbot = create_agent(
    model=llm,
    tools=[],
    system_prompt="너는 친절한 한국어 챗봇이야.",
    checkpointer=InMemorySaver(),   # 대화 기록 저장소 (RAM)
)

cfg = {"configurable": {"thread_id": "1"}}   # 대화방 번호

for q in ["안녕하세요 챗봇님", "나는 홍길동입니다", "내 이름이 뭐였죠?"]:
    result = chatbot.invoke({"messages": [("human", q)]}, config=cfg)
    print(f"Q: {q}\nA: {result['messages'][-1].text}\n")

저장된 대화 기록은 `get_state()`로 확인합니다.

In [ ]:
state = chatbot.get_state(cfg)
for msg in state.values["messages"]:
    print(f"[{msg.type}] {msg.text}")

다른 `thread_id`로 호출하면 **완전히 독립된 대화방**이 열립니다.

In [ ]:
cfg2 = {"configurable": {"thread_id": "2"}}
result = chatbot.invoke({"messages": [("human", "내 이름이 뭐였죠?")]}, config=cfg2)
print(result["messages"][-1].text)   # thread 2에는 이름 정보가 없습니다.

### 해 보기

각각의 대화 기록을 갖는 3명의 대화를 만들어 보세요.

- user1: "나는 일본 여행을 준비 중이야." → "오사카와 교토를 방문할 예정이야." → "내가 어디를 여행하려고 했지?"
- user2: "우리 백화점 이름은 행복백화점이야." → "이번 달 할인 행사는 여름 정기세일이야." → "우리 백화점 이름이 뭐였지?"
- user3: "우리 식당 이름은 맛있는집이야." → "대표 메뉴는 김치찌개야." → "우리 대표 메뉴가 뭐였지?"

In [ ]:
# 여기에 실습 코드를 작성하세요.

## 7. Window Memory - 최근 k개의 대화만 유지

`trim_messages`로 **최근 메시지 k개만 남기고** 나머지는 잘라 냅니다.

| 파라미터 | 의미 |
|---|---|
| `strategy="last"` | 뒤에서부터(최근 것부터) 남김 |
| `token_counter=len` | 토큰 대신 **메시지 개수**로 계산 |
| `max_tokens` | 남길 최대 개수 |
| `include_system=True` | 시스템 메시지는 항상 유지 |
| `start_on="human"` | 잘린 뒤 첫 메시지가 사용자 질문이 되도록 맞춤 |

In [ ]:
from langchain_core.messages import trim_messages

WINDOW_K = 1   # 최근 1턴(질문+답변)만 기억

def window(messages, k=WINDOW_K):
    return trim_messages(
        messages,
        strategy="last",
        token_counter=len,
        max_tokens=k * 2 + 2,   # system 1개 + 최근 k턴(2개씩) + 새 질문 1개
        include_system=True,
        start_on="human",
    )

history = [("system", "당신은 서울 여행 가이드입니다. 짧게 답하세요.")]
questions = [
    "서울을 관광하고 있습니다. 서울 음식점 한 곳만 알려 주세요.",
    "그곳의 제일 인기 있는 메뉴는 무엇인가요?",
    "서울역에서 그곳까지 대중교통으로 가려면 어떻게 가야 하나요?",
]

for q in questions:
    history.append(("human", q))
    response = llm.invoke(window(history))   # 최근 k턴만 전달
    history.append(("ai", response.content))
    print(f"Q: {q}\nA: {response.content}\n")

print("전체 기록 메시지 수 :", len(history))
print("LLM에 전달되는 메시지 수:", len(window(history + [("human", "다음 질문")])))

> k가 작으면 비용은 줄지만, 3번째 질문의 "그곳"처럼 **오래전 맥락을 잊을 수 있습니다.** `WINDOW_K` 값을 바꿔 결과를 비교해 보세요.

## 8. Summary Memory - 대화 내용 요약

대화가 길어지면 **오래된 대화를 LLM으로 요약**하고, 최근 대화만 원문으로 유지합니다.
LangChain 1.x에서는 `create_agent`의 **`SummarizationMiddleware`** 로 자동 처리합니다.

| 파라미터 | 의미 |
|---|---|
| `model` | 요약에 사용할 모델 |
| `trigger=("messages", 6)` | 메시지가 6개 이상이면 요약 실행 |
| `keep=("messages", 2)` | 최근 메시지 2개는 원문 유지 |

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

summary_bot = create_agent(
    model=llm,
    tools=[],
    system_prompt="당신은 서울 여행 가이드입니다. 짧게 답하세요.",
    middleware=[
        SummarizationMiddleware(model=llm, trigger=("messages", 6), keep=("messages", 2)),
    ],
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "summary-1"}}
questions = [
    "안녕하세요, 저는 서울을 처음 여행합니다.",
    "서울 음식점 한 곳만 알려 주세요.",
    "그곳의 제일 인기 있는 메뉴는 무엇인가요?",
    "서울역에서 그곳까지 대중교통으로 가려면 어떻게 가야 하나요?",
]
for q in questions:
    result = summary_bot.invoke({"messages": [("human", q)]}, config=cfg)
    print(f"Q: {q}\nA: {result['messages'][-1].text}\n")

In [ ]:
# 저장된 기록 확인: 앞부분이 요약 메시지로 바뀌어 있습니다.
for msg in summary_bot.get_state(cfg).values["messages"]:
    print(f"[{msg.type}] {msg.text[:150]}")
    print()

> 요약에도 LLM 호출이 필요하므로 **추가 비용**이 발생하고, 요약 과정에서 세부 정보가 빠질 수 있습니다.

## 9. Token Usage - 토큰 사용량과 비용 확인

- LangChain의 응답(`AIMessage`)에는 `usage_metadata`로 토큰 사용량이 들어 있습니다.
- 체인이나 에이전트처럼 여러 번 호출할 때는 `get_usage_metadata_callback()`으로 합계를 확인합니다.

| 항목 | 의미 |
|---|---|
| `input_tokens` | 프롬프트(입력)로 사용된 토큰 수 |
| `output_tokens` | 생성된 응답 토큰 수 |
| `total_tokens` | 전체 토큰 수 |

In [ ]:
response = llm.invoke("서울 광장시장에서 가장 맛있는 길거리 음식은 무엇인가요? 한 문장으로 답해 주세요.")
print(response.content)
print(response.usage_metadata)

In [ ]:
from langchain_core.callbacks import get_usage_metadata_callback

# gpt-6-luna 표준 요금 (USD / 100만 토큰) - 수업 전 공식 가격 문서에서 확인하세요.
PRICE_INPUT = 0.10
PRICE_OUTPUT = 0.50

with get_usage_metadata_callback() as cb:
    llm.invoke("서울의 대표 음식 3가지를 알려 주세요.")
    llm.invoke("부산의 대표 음식 3가지를 알려 주세요.")

for model_name, usage in cb.usage_metadata.items():
    cost = usage["input_tokens"] / 1e6 * PRICE_INPUT + usage["output_tokens"] / 1e6 * PRICE_OUTPUT
    print(model_name)
    print(f"  Input Tokens : {usage['input_tokens']}")
    print(f"  Output Tokens: {usage['output_tokens']}")
    print(f"  Total Tokens : {usage['total_tokens']}")
    print(f"  예상 비용(USD): ${cost:.6f}")